# California Housing Price Prediction: OLS vs Ridge vs Lasso

**Goal:** Predict median house values and compare OLS, Ridge (L2), and Lasso (L1) regression using RMSE, MAE, and R².

This notebook covers data loading, missing-value imputation, engineered features, correlation heatmap, 80/20 split, numeric scaling, categorical encoding, GridSearchCV, predicted-vs-actual plots, and residual plots. Run cells from top to bottom. Internet access is needed to load the CSV.

## 1. Import libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load dataset

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv"
df = pd.read_csv(DATA_URL)
print("Dataset shape:", df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values"))

## 3. Feature engineering

In [ ]:
def add_features(data):
    data = data.copy()
    data["rooms_per_household"] = data["total_rooms"] / data["households"].replace(0, np.nan)
    data["bedrooms_per_room"] = data["total_bedrooms"] / data["total_rooms"].replace(0, np.nan)
    data["population_per_household"] = data["population"] / data["households"].replace(0, np.nan)
    data["income_x_rooms"] = data["median_income"] * data["rooms_per_household"]
    return data

df_features = add_features(df)
display(df_features.head())
display(df_features.describe().T)

## 4. Correlation heatmap and multicollinearity

The heatmap helps identify highly correlated numeric features. Ridge and Lasso regularization later help control unstable coefficients caused by multicollinearity. Correlation does not prove causation.

In [ ]:
numeric_corr = df_features.select_dtypes(include=np.number).copy()
numeric_corr = numeric_corr.fillna(numeric_corr.median(numeric_only=True))
plt.figure(figsize=(13, 9))
sns.heatmap(numeric_corr.corr(), cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
display(numeric_corr.corr()["median_house_value"].drop("median_house_value").sort_values().to_frame("correlation_with_target"))

## 5. Split features and target (80/20)

In [ ]:
X = df_features.drop(columns=["median_house_value"])
y = df_features["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
print("Training rows:", len(X_train), "| Testing rows:", len(X_test))

## 6. Preprocessing: impute, scale, and encode

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 7. Train OLS, Ridge and Lasso; tune alpha with GridSearchCV

In [ ]:
ols = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
ols.fit(X_train, y_train)

ridge = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])
ridge_search = GridSearchCV(
    ridge, {"model__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
ridge_search.fit(X_train, y_train)

lasso = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=20000))
])
lasso_search = GridSearchCV(
    lasso, {"model__alpha": [0.001, 0.01, 0.1, 1.0, 10.0]},
    cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
lasso_search.fit(X_train, y_train)

models = {
    "OLS": ols,
    "Ridge (tuned)": ridge_search.best_estimator_,
    "Lasso (tuned)": lasso_search.best_estimator_
}
print("Best Ridge parameters:", ridge_search.best_params_)
print("Best Lasso parameters:", lasso_search.best_params_)

## 8. Compare test-set metrics

In [ ]:
results = []
predictions = {}
for name, model in models.items():
    pred = model.predict(X_test)
    predictions[name] = pred
    results.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "MAE": mean_absolute_error(y_test, pred),
        "R2": r2_score(y_test, pred)
    })
results_df = pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(3))

## 9. Predicted vs actual plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True, sharey=True)
for ax, (name, pred) in zip(axes, predictions.items()):
    ax.scatter(y_test, pred, alpha=0.25, s=10)
    lo = min(float(y_test.min()), float(np.min(pred)))
    hi = max(float(y_test.max()), float(np.max(pred)))
    ax.plot([lo, hi], [lo, hi], "r--")
    ax.set_title(name)
    ax.set_xlabel("Actual median house value")
    ax.set_ylabel("Predicted median house value")
fig.suptitle("Predicted vs Actual", y=1.03)
plt.tight_layout()
plt.show()

## 10. Residual distribution curves

Residual = actual value − predicted value. Residuals centered around zero are generally desirable; use this plot together with the other evaluation metrics.

In [ ]:
plt.figure(figsize=(11, 6))
for name, pred in predictions.items():
    residuals = y_test.to_numpy() - pred
    sns.kdeplot(residuals, label=name)
plt.axvline(0, color="black", linestyle="--")
plt.title("Residual Distribution")
plt.xlabel("Residual (actual − predicted)")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Residuals vs predicted values

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for ax, (name, pred) in zip(axes, predictions.items()):
    residuals = y_test.to_numpy() - pred
    ax.scatter(pred, residuals, alpha=0.25, s=10)
    ax.axhline(0, color="red", linestyle="--")
    ax.set_title(name)
    ax.set_xlabel("Predicted value")
    ax.set_ylabel("Residual")
plt.tight_layout()
plt.show()

## 12. Conclusion

In [ ]:
best = results_df.iloc[0]
print("Lowest test RMSE model:", best["Model"])
print(f"RMSE: {best['RMSE']:.2f}")
print(f"MAE: {best['MAE']:.2f}")
print(f"R²: {best['R2']:.4f}")
display(results_df.round(3))
print("Lower RMSE/MAE means smaller errors; higher R² means more target variation is explained.")

## Notes
- Numeric missing values are imputed with the median; categorical missing values with the most frequent category.
- Numeric features are standardized and `ocean_proximity` is one-hot encoded.
- Ratio/interaction-style features are included.
- Ridge and Lasso alpha values are selected using 5-fold GridSearchCV.
- Only report metric values in your README after running the notebook and checking its results.